In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

# 1. Load Data 
df_energy = pd.read_csv("data/energy_dataset.csv")

# 2. Parse the 'time' column to datetime and sort 
# (Make sure to use the exact name of your time column, usually 'time' in this Kaggle dataset)
df_energy['time'] = pd.to_datetime(df_energy['time'], utc=True)
df_energy = df_energy.sort_values('time')

# Fill NaNs with linear interpolation
df_energy = df_energy.infer_objects(copy=False).interpolate(method="linear")

# 3. Time Series Train/Test Split by Year
# Everything before 2018 is train/validation, 2018 is the test set
df_train_val = df_energy[df_energy['time'].dt.year < 2018]
df_test = df_energy[df_energy['time'].dt.year == 2018]

# 4. Extract series from the 2018 TEST SET ONLY
y_true_test = df_test["total load actual"]
y_pred_forecast_test = df_test["total load forecast"]

# Ensure no NaNs remain before metric calculation
valid_idx = y_true_test.notna() & y_pred_forecast_test.notna()
y_true_clean = y_true_test[valid_idx]
y_pred_forecast_clean = y_pred_forecast_test[valid_idx]

### TSO Official Forecast Error (ON 2018 TEST SET)
tso_mae = mean_absolute_error(y_true_clean, y_pred_forecast_clean)
tso_mape = mean_absolute_percentage_error(y_true_clean, y_pred_forecast_clean)
tso_rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_forecast_clean))

### Summary
ref_error = pd.DataFrame({
    "Method": ["TSO Forecast (2018 Test Set)"],
    "MAE (MWh)": [tso_mae],
    "MAPE": [tso_mape],
    "RMSE (MWh)": [tso_rmse]
})

print("\nSummary of Baseline Errors (2018 Hold-out Period):")
print(ref_error.to_string(index=False))


Summary of Baseline Errors (2018 Hold-out Period):
                      Method  MAE (MWh)     MAPE  RMSE (MWh)
TSO Forecast (2018 Test Set)  269.85004 0.009257  389.322653
